# Chapter 8: Introduction and Overview
*From: Database Internals — Part II opener, Distributed Systems*

## TL;DR

This chapter is the **vocabulary and mental model** for everything that follows in Part II. It doesn't design a system; it names the failures and impossibilities that make distributed system design *different from* single-node programming, and it builds the abstractions that later chapters (failure detectors, consensus, replication) stand on.

- **Distributed computing is defined by what goes wrong, not by what it does.** The network drops, reorders, and delays messages; processes crash; clocks drift; queues fill; nodes come back with stale state. The "happy path" is a special case.
- **Three impossibility results fence in the design space.** Two Generals (no guaranteed agreement over a lossy asynchronous channel via finite exchange), FLP (no deterministic asynchronous consensus with even one crash), and the lower bound that failure detection requires *some* synchrony assumption. Every real protocol weakens one of these to make progress.
- **Communication is a ladder of abstractions.** Fair-loss link → add ACKs + seqnos → stubborn link (retransmit forever) → combine → perfect link (reliable delivery, no duplication, no creation). This ladder is built from exactly three mechanisms: acknowledgments, retransmits, and deduplication.
- **The four failure models form a strict hierarchy.** Crash-stop ⊂ crash-recovery ⊂ omission ⊂ Byzantine. Each level subsumes the previous one and demands more redundancy — a crash-tolerant quorum needs `2f + 1` nodes, a Byzantine-tolerant one needs `3f + 1`.
- **Real systems live in partial synchrony.** Pure async can't solve consensus (FLP); pure sync is unrealistic (links don't have hard upper bounds on delay). Partial synchrony — bounds hold *eventually* or *most of the time* — is where Raft, Paxos, and every practical failure detector operate.
- **Cascading failure is the default.** A slow node causes retries, retries cause overload, overload causes more slow nodes. The remedies (backpressure, circuit breakers, exponential backoff with jitter, checksumming) are themselves distributed-systems primitives, not afterthoughts.

## What This Chapter Establishes

Unlike design chapters, this one is a glossary with teeth. Subsequent chapters assume you know these terms precisely. The mental models introduced:

| # | Concept | One-line summary |
|---|---------|------------------|
| 1 | Fallacies of Distributed Computing | Deutsch's eight originals (reliable network, zero latency, infinite bandwidth, secure network, stable topology, one administrator, zero transport cost, homogeneous network) plus the chapter's five extensions (instant processing, infinite queue capacity, synced clocks, local = remote, strongly consistent state). |
| 2 | Partial Failures and Cascading Failures | Why partial states diverge across observers, and the chain reactions (retry storms, catch-up storms, corruption spread) that turn one hiccup into an outage without backpressure / circuit breakers / backoff+jitter / checksums. |
| 3 | Link Abstractions and Delivery Semantics | The ladder from fair-loss to stubborn to perfect links, built from ACKs, retransmits, seqnos, reorder buffers, and dedup — plus what at-most-once / at-least-once / exactly-once actually mean. |
| 4 | Two Generals' Problem | The proof that asynchronous lossy channels cannot deliver common knowledge via any finite ACK exchange, and therefore cannot guarantee agreement. |
| 5 | FLP Impossibility and Consensus Properties | No deterministic async protocol guarantees consensus (agreement + validity + termination) in bounded time with even a single possible crash — so real systems weaken async. |
| 6 | System Synchrony Models | Async (no bounds) vs sync (hard bounds) vs partial sync (bounds hold eventually or mostly); what each model can solve. |
| 7 | Failure Models | Crash-stop, crash-recovery, omission, Byzantine — a strict hierarchy of increasingly adversarial behaviour and of increasingly expensive tolerance. |

---

## Quantitative Intuitions

The chapter is conceptual, but the conclusions become intuitive only when you see the numbers. The code cells below are all stdlib-only and runnable — each one isolates one piece of distributed-systems reasoning:

1. **Fair-loss link math** — how many retries to reach a target delivery probability over a 1% lossy link.
2. **Two Generals' common-knowledge regress** — the probability that both sides mutually know, after N rounds of ACK-of-ACK.
3. **Partial-synchrony timeout sizing** — choosing a failure-detector timeout from a p99 RTT distribution (false-positive vs detection-latency trade-off).
4. **Gossip heartbeat bandwidth** — per-node and cluster-wide traffic as cluster size scales.
5. **Quorum math for crash vs Byzantine** — why crash tolerance needs `2f + 1` and Byzantine tolerance needs `3f + 1`, and what that costs in absolute node count.

### 1. Fair-loss link: retries to reach 99.9% delivery

A fair-loss link guarantees that if you retransmit infinitely, the message eventually arrives. But how many retries does it take to make "eventually" feel like "now"? If each transmission drops with probability `p`, after `k` independent attempts the message is still undelivered with probability `p**k`. Inverting gives the minimum `k` for any target delivery probability.

In [ ]:
import math

def retries_needed(p_drop: float, target_delivered: float) -> int:
    """Min retransmissions to reach target delivery probability on a fair-loss link."""
    # P(lost after k tries) = p_drop**k ; we want p_drop**k <= 1 - target
    return math.ceil(math.log(1 - target_delivered) / math.log(p_drop))

def expected_attempts(p_drop: float) -> float:
    """Mean attempts until first success — geometric distribution."""
    return 1 / (1 - p_drop)

drop_rates = [0.01, 0.05, 0.20, 0.50]
targets    = [0.99, 0.999, 0.99999]

print(f"{'p_drop':>8} {'E[attempts]':>12}   " + "  ".join(f"p>={t:.5f}" for t in targets))
for p in drop_rates:
    row = f"{p:>8.2%} {expected_attempts(p):>12.2f}   "
    row += "  ".join(f"{retries_needed(p, t):>8d}" for t in targets)
    print(row)

# Latency implication: if RTT = 50 ms and we time out after 2*RTT before retrying,
# reaching 99.9% at 20% drop takes this many retries and this long:
rtt_ms        = 50
timeout_mult  = 2
p_drop        = 0.20
target        = 0.999
k             = retries_needed(p_drop, target)
tail_latency  = k * timeout_mult * rtt_ms
print(f"\nAt p_drop={p_drop:.0%}, target={target}: {k} retries, ~{tail_latency} ms tail latency.")
print("=> 'eventually delivered' != 'delivered in time'. Fair-loss is a correctness guarantee, not a latency one.")

### 2. Two Generals' problem: common-knowledge regress

Each messenger crosses enemy territory and gets through with probability `p_through`. After general A sends MSG, B replies ACK, A replies ACK(ACK), and so on, each side's confidence in mutual knowledge grows — but never reaches 1. The last sender is always one unacknowledged message away from certainty.

In [ ]:
def mutual_knowledge_after_rounds(p_through: float, rounds: int) -> float:
    """Probability that both sides have progressed through `rounds` messenger crossings.

    Each round is one messenger crossing the lines. For both sides to have completed
    `rounds` exchanges, ALL `rounds` messengers must have succeeded.
    """
    return p_through ** rounds

for p in [0.9, 0.7, 0.5]:
    print(f"\nMessenger success per crossing: {p:.0%}")
    for r in [1, 2, 4, 8, 16]:
        conf = mutual_knowledge_after_rounds(p, r)
        print(f"  After {r:>2} round(s) of ACK/ACK-of-ACK: P(both caught up) = {conf:.4f}")

# The chapter's point: no matter how large r gets, the LAST sender cannot distinguish
# "my ACK got through" from "my ACK was lost". Confidence asymptotes; it never reaches 1.
print("\nNotice: doubling rounds does NOT square the confidence — it just adds one more coin flip.")
print("In the limit, P -> 0, not 1. Adding rounds burns latency without buying certainty.")

### 3. Partial-synchrony timeout sizing

In a partially-synchronous model, RTT is *usually* bounded but can spike. A failure-detector timeout that is too short produces false positives (healthy nodes marked dead → unnecessary failover, split brain risk). Too long and real crashes take forever to detect. The sweet spot is usually a few multiples of the observed p99 RTT.

In [ ]:
import math, random, statistics

random.seed(42)

# Simulate an RTT distribution: lognormal-ish, median 20 ms, fat tail.
def sample_rtt_ms() -> float:
    # median ~20 ms, p99 ~150 ms, occasional 500+ ms spikes
    return max(1.0, random.lognormvariate(mu=math.log(20), sigma=0.8))

samples = sorted(sample_rtt_ms() for _ in range(20_000))
def pct(p): return samples[int(len(samples) * p)]

p50, p90, p99, p999 = pct(0.50), pct(0.90), pct(0.99), pct(0.999)
print(f"RTT distribution:  p50={p50:.0f} ms  p90={p90:.0f} ms  p99={p99:.0f} ms  p999={p999:.0f} ms")

# Try candidate timeouts as multiples of p99.
# False-positive rate = fraction of healthy probes exceeding the timeout.
# Detection latency (worst case for a real crash) ~ timeout + one probe interval.
probe_interval_ms = 200
print(f"\n{'timeout_mult':>12} {'timeout_ms':>12} {'FP_rate':>10} {'detect_ms_real_crash':>22}")
for mult in [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
    timeout_ms = mult * p99
    fp_rate = sum(1 for s in samples if s > timeout_ms) / len(samples)
    detect_ms = timeout_ms + probe_interval_ms
    print(f"{mult:>12.1f} {timeout_ms:>12.0f} {fp_rate:>10.4f} {detect_ms:>22.0f}")

print("\nRule of thumb: pick the smallest multiple where FP_rate falls below the tolerable")
print("false-failover budget (often ~1e-3 to 1e-4 per probe). The cost is slower real-crash")
print("detection — exactly the FLP/partial-synchrony trade-off stated in math terms.")

### 4. Gossip heartbeat bandwidth at cluster scale

If every node heartbeats to every other node every second (all-to-all), the cluster-wide packet rate is `N * (N-1)` per second — quadratic. Gossip protocols fan out to a constant `k` peers per round, making per-node traffic linear in `k` and cluster-wide traffic linear in `N`. The numbers make the case for gossip instantly obvious at N=1000.

In [ ]:
def all_to_all_bw(n_nodes: int, heartbeat_bytes: int, hz: float) -> tuple[int, int]:
    """Return (per_node_bytes_per_sec, cluster_bytes_per_sec) for all-to-all heartbeating."""
    per_node = (n_nodes - 1) * heartbeat_bytes * hz
    cluster  = n_nodes * per_node
    return per_node, cluster

def gossip_bw(n_nodes: int, fanout: int, heartbeat_bytes: int, hz: float) -> tuple[int, int]:
    """Each node sends to `fanout` random peers per round."""
    per_node = fanout * heartbeat_bytes * hz
    cluster  = n_nodes * per_node
    return per_node, cluster

heartbeat_bytes = 256   # small UDP-ish packet: identity + view digest + seqno
hz              = 1.0   # once per second
fanout          = 3     # typical SWIM / gossip fanout

print(f"{'N':>6} | {'all2all/node':>14} {'all2all/cluster':>18} | {'gossip/node':>12} {'gossip/cluster':>16}")
print("-" * 80)
for n in [10, 100, 1_000, 10_000]:
    a_per, a_cluster = all_to_all_bw(n, heartbeat_bytes, hz)
    g_per, g_cluster = gossip_bw(n, fanout, heartbeat_bytes, hz)
    def fmt(b): return f"{b/1024:>10.1f} KB/s" if b < 1024**2 else f"{b/1024**2:>10.1f} MB/s"
    print(f"{n:>6} | {fmt(a_per):>14} {fmt(a_cluster):>18} | {fmt(g_per):>12} {fmt(g_cluster):>16}")

print("\nAt N=1000, all-to-all burns ~250 MB/s cluster-wide just on heartbeats.")
print("Gossip flatlines at ~750 KB/node/sec regardless of N. This is why failure")
print("detection at scale is gossip-based, not broadcast-based (see ch. 9).")

### 5. Quorum math: crash-tolerance vs Byzantine-tolerance

To tolerate `f` **crash** failures a protocol needs `2f + 1` nodes (so any majority has at least one correct intersection). To tolerate `f` **Byzantine** failures it needs `3f + 1` (so any two quorums intersect in at least `f + 1` correct nodes, defeating equivocation). The absolute cost difference is striking.

In [ ]:
def min_nodes_crash(f: int) -> int:    return 2 * f + 1
def min_nodes_byzantine(f: int) -> int: return 3 * f + 1

def quorum_crash(n: int) -> int:     return n // 2 + 1        # simple majority
def quorum_byzantine(n: int) -> int: return (2 * n) // 3 + 1  # >= 2/3

print(f"{'f':>4} | {'crash_n':>8} {'crash_Q':>8} | {'byz_n':>8} {'byz_Q':>8} | {'extra_nodes_for_BFT':>22}")
print("-" * 70)
for f in [1, 2, 3, 5, 10, 33]:
    cn, bn = min_nodes_crash(f), min_nodes_byzantine(f)
    cq, bq = quorum_crash(cn), quorum_byzantine(bn)
    print(f"{f:>4} | {cn:>8} {cq:>8} | {bn:>8} {bq:>8} | {bn - cn:>22}")

print("\nTolerating 10 failures: crash needs 21 nodes, Byzantine needs 31 — 48% more hardware")
print("for every round of every decision. This is why classical DBs assume crash-stop and")
print("only cryptocurrency / aerospace systems pay the Byzantine premium.")

---

## 1. Fallacies of Distributed Computing

Deutsch's 1994 list is the industry's standing reminder that every happy-path assumption is a latent bug. Each fallacy looks innocuous until the network, the hardware, or the ops team makes it false — and then the failure mode is usually "system silently does the wrong thing," not "system crashes loudly." The original eight target the network (reliability, latency, bandwidth, security, topology, administration, transport cost, heterogeneity). The chapter then adds five more that target the node and the system (instant processing, infinite queue capacity, synced clocks, local == remote, strongly consistent state).

```mermaid
graph TB
    subgraph Orig[Deutsch's original eight - target the wire]
        F1[Network is reliable<br/>-> silent packet loss]
        F2[Latency is zero<br/>-> timeouts too aggressive]
        F3[Bandwidth is infinite<br/>-> queues explode]
        F4[Network is secure<br/>-> MITM, forged messages]
        F5[Topology is stable<br/>-> stale service discovery]
        F6[One administrator<br/>-> uncoordinated config drift]
        F7[Transport cost is zero<br/>-> egress bills, CPU tax]
        F8[Homogeneous network<br/>-> protocol mismatches]
    end
    subgraph Ext[Chapter extensions - node and system level]
        E1[Processing is instant<br/>-> p99 tail latency]
        E2[Queue capacity is infinite<br/>-> OOM, unbounded latency]
        E3[Clocks are in sync<br/>-> ordering bugs]
        E4[Local == remote<br/>-> hidden remote cost]
        E5[State is strongly consistent<br/>-> schema/ring divergence]
    end
    Orig --> Cascade[Retries + timeouts layered on false assumptions<br/>= cascading failures]
    Ext --> Cascade
```

### Fallacies, failures, and mitigations

| Fallacy | What silently breaks | Typical mitigation |
|---------|----------------------|--------------------|
| Network is reliable | Retries mask fallback logic; partitions cause split-brain | Explicit timeouts, idempotent ops, circuit breakers |
| Latency is zero | API assumes local call semantics; blocking loops | Async APIs; batch RPCs; colocate data and compute |
| Bandwidth is infinite | Replication storms, catch-up streams DOS the cluster | Backpressure, rate limiting, separate replication NIC |
| Clocks are in sync | Timestamp-ordered ops diverge across nodes | Hybrid logical clocks (HLC), TrueTime-style uncertainty intervals, Lamport timestamps |
| Processing is instant | Request queues pile up; p99 blows out | Bounded queues, load shedding, adaptive concurrency |
| Topology is stable | Clients talk to departed nodes; schema/ring views diverge | Membership protocols, versioned cluster views, epoch numbers |
| Network is secure | Message forgery, replay | TLS, message authentication codes, mutual auth |
| One administrator | Uncoordinated config changes break invariants | Config stores with consensus (etcd/zk), change review, canarying |

---

## 2. Partial Failures and Cascading Failures

A **partial failure** is the condition where some participants see a node as alive and others see it as dead — the same event looks different from different observation points. The failure is neither total (the rest of the cluster works) nor clean (there is no single truth about what happened). That ambiguity is the source of most distributed-systems bugs: every design must either tolerate it explicitly or impose an expensive coordination protocol to hide it.

Cascading failure is the follow-on dynamic: one slow or failed node induces retries from its clients; the retries add load to the already-stressed node and its replicas; the added load slows them down; more clients mark them failed; and so on. Left unchecked the local failure propagates until it consumes a significant fraction of the cluster. The three standard brakes on this feedback loop are **circuit breakers** (stop calling the failing service for a cool-down window), **exponential backoff with jitter** (spread retries in time so a thundering herd doesn't re-synchronize on each backoff tick), and **checksumming / validation** (prevent corrupted data from being propagated through normal replication).

```mermaid
graph LR
    Slow[Node N gets slow] --> Timeout[Clients hit timeout]
    Timeout --> Retry[Retry in tight loop]
    Retry --> MoreLoad[Load on N increases]
    MoreLoad --> Slow
    Retry --> Neighbor[Neighbors pick up fallback traffic]
    Neighbor --> NeighborSlow[Neighbors slow down]
    NeighborSlow --> Outage[Cluster-wide degradation]

    Retry -. circuit breaker opens .-> Stop[Stop calling N]
    Retry -. backoff + jitter .-> Spread[Retries spread in time]
    Replication[Replication stream] -. checksum fail .-> Reject[Reject corrupted page]

    Stop --> Recover[N has room to recover]
    Spread --> Recover
```

### Cascading-failure remedies

| Mechanism | What it prevents | Cost |
|-----------|------------------|------|
| Circuit breaker | Retry amplification against a failing dependency | Some healthy requests rejected during cool-down |
| Exponential backoff | Tight retry loops | Worse best-case latency on transient errors |
| Jitter | Retry synchronization across clients | Slight variance in retry timing (usually free) |
| Backpressure (bounded queues) | Unbounded queue growth → OOM → crash | Load shedding; upstream sees errors sooner |
| Checksumming / validation | Corrupted data replicating silently | CPU cost on every message / page |
| Coordinated execution (plans) | Peers independently overloading the same hotspot | Coordinator becomes another component to make reliable |

---

## 3. Link Abstractions and Delivery Semantics

Links are the primitive over which every higher-level distributed protocol is built. The chapter constructs three abstractions in layers, each stronger than the last:

- **Fair-loss link.** If the sender keeps retransmitting forever and both parties are correct, the message is eventually delivered. Finite duplication (nothing is delivered infinitely many times). No creation (nothing is delivered that wasn't sent). Think UDP without any application protocol on top.
- **Stubborn link.** Built by adding retransmits. The sender resends until... well, forever, since it has no ACKs to tell it to stop. Impractical alone but the right stepping stone.
- **Perfect link.** Built by combining retransmits with acknowledgments and sequence numbers. Gives **reliable delivery** (every sent message eventually delivered once) + **no duplication** (dedup on seqno) + **no creation**. TCP (within a single session) approximates this.

Delivery semantics are the application-visible consequence:
- **At-most-once**: send and don't care — may be lost, never duplicated. Cheap, sufficient for telemetry.
- **At-least-once**: retry until ACKed — may be duplicated. Requires idempotent handlers or application-level dedup.
- **Exactly-once-delivery** is impossible at the link level, but **exactly-once-processing** is achievable: transmit at-least-once and deduplicate at the receiver on message identity. This is the only honest reading of "exactly once."

```mermaid
graph TB
    FL[Fair-loss link<br/>no ACK, may lose / reorder / dup<br/>bounded by 'no creation']
    FL -->|+ retransmits| SL[Stubborn link<br/>keeps resending forever]
    FL -->|+ ACK + seqno + dedup| PL[Perfect link<br/>reliable delivery<br/>no duplication<br/>no creation]
    SL -->|+ ACK + seqno + dedup| PL

    PL --> Sem1[at-most-once<br/>send and forget]
    PL --> Sem2[at-least-once<br/>retry until ACKed]
    PL --> Sem3[exactly-once processing<br/>at-least-once + dedup<br/>not 'exactly-once delivery']
```

### Link abstractions head-to-head

| Property | Fair-loss | Stubborn | Perfect |
|----------|:---------:|:--------:|:-------:|
| Fair loss (retransmits eventually succeed) | yes | yes | yes |
| Finite duplication | yes | **no** (unbounded retries) | yes |
| No creation | yes | yes | yes |
| Reliable delivery (exactly one delivery per send) | no | yes (eventually) | **yes** |
| No duplication at receiver | — | no | **yes** (via seqno dedup) |
| In-order delivery | no | no | **yes** (via reorder buffer) |
| Real-world analogue | UDP | retry-forever RPC | TCP (per session) |

### Delivery-semantics summary

| Semantic | How implemented | Requires at receiver | Good for |
|----------|-----------------|----------------------|----------|
| At-most-once | Send, no retry | Nothing | Metrics, telemetry, UDP broadcasts |
| At-least-once | Retry until ACK | Idempotent handler or dedup | General RPC, message queues |
| Exactly-once processing | At-least-once + receiver-side dedup on message ID | Dedup table keyed by sender seqno or message hash | Payments, ledgers, state machines |

---

## 4. Two Generals' Problem

Two armies on opposite sides of a city must attack simultaneously or lose. They communicate by messenger across enemy territory; every messenger may be captured. General A sends MSG("attack at dawn"). He doesn't know if it arrived. B, on receipt, sends ACK. B doesn't know if the ACK arrived. A could reply ACK(ACK), but then A doesn't know if *that* got through. Every message is answerable only by another message that needs its own acknowledgment — and the last sender is always uncertain.

The problem is impossible without timing assumptions: communication is asynchronous, the channel is lossy, and no finite protocol bridges the gap. Crucially, this result is independent of link quality — even a "perfect" TCP-like channel within a single session cannot guarantee the *application* agrees on whether a message was received, because the session itself can drop, the process can crash after delivery but before processing, and so on.

The practical consequence: **common knowledge** (both sides know X, and both sides know that both sides know X, ad infinitum) is not achievable over asynchronous lossy channels. All real systems settle for **approximate** or **weak** common knowledge — "both sides know X with high probability" — usually obtained by timing out, assuming the peer knows, and retrying if the assumption was wrong.

```mermaid
sequenceDiagram
    participant A as General A
    participant B as General B
    A->>B: MSG("attack at dawn")
    Note right of A: Did it arrive?
    B->>A: ACK(MSG)
    Note left of B: Did my ACK arrive?
    A->>B: ACK(ACK(MSG))
    Note right of A: Did my second ACK arrive?
    B->>A: ACK(ACK(ACK(MSG)))
    Note left of B: ...ad infinitum
    Note over A,B: Last sender always uncertain.<br/>No finite exchange produces common knowledge.
```

### Trade-off: how real systems cope with Two Generals

| Strategy | What it gives up | What it gains |
|----------|------------------|---------------|
| Accept uncertainty (fire-and-forget) | Agreement guarantee | Simplicity, at-most-once latency |
| Retry + dedup (at-least-once processing) | Exactly-once *delivery* illusion | Eventual mutual processing |
| Add timing assumptions (partial sync) | Pure-async semantics | Ability to time out, declare failure, proceed |
| Consensus protocol (Paxos/Raft) | Low latency, low node count | Bounded-decision agreement in partial sync |
| Atomic commit (2PC) with coordinator | Availability during coordinator failure | Bounded agreement *if coordinator lives* |

---

## 5. FLP Impossibility and Consensus

Fischer, Lynch, and Paterson (1985) proved: **no deterministic consensus protocol in a fully asynchronous system can guarantee termination if even a single process may crash silently**. Consensus here means all correct processes decide on some proposed value such that:

- **Agreement** — every correct process decides the same value.
- **Validity** — the decided value was proposed by some process (not a default invented by the algorithm).
- **Termination** — every correct process eventually decides.

You can have any two of these under pure async + possible crash; you cannot have all three. This is NOT a statement that consensus is impossible — it's a statement that the pure-async + deterministic + crash-tolerant + always-terminating combination is impossible. Real protocols relax one assumption:

- **Partial synchrony** (Raft, Paxos) — add weak timing bounds that hold eventually. Termination is guaranteed only when synchrony holds.
- **Randomization** (Ben-Or, HoneyBadgerBFT) — make progress probabilistic; termination is with probability 1, not certainty.
- **Failure detectors** (Chandra-Toueg) — assume an oracle (usually implemented via timeouts) that eventually correctly reports failures.

```mermaid
graph LR
    subgraph Triangle[Consensus: pick any two under pure async + crash]
        A[Agreement<br/>all correct processes<br/>decide same value]
        V[Validity<br/>decided value was<br/>proposed by someone]
        T[Termination<br/>every correct process<br/>eventually decides]
    end
    A --- V
    V --- T
    T --- A

    Triangle --> Escape[Real-world escape hatches]
    Escape --> E1[Partial synchrony<br/>Raft, Paxos]
    Escape --> E2[Randomization<br/>Ben-Or, HoneyBadger]
    Escape --> E3[Failure detectors<br/>Chandra-Toueg]
```

### Consensus properties and what breaks where

| Property | Held by | Broken by |
|----------|---------|-----------|
| Agreement | Always required for correctness | Split brain, network partition + no quorum discipline |
| Validity | Always required | A "default value" oracle that ignores proposals |
| Termination | Easy in sync; hard in async | FLP — an adversary can stall forever by delaying one message |

---

## 6. System Synchrony Models

Synchrony is the timing assumption the algorithm is allowed to make. It ranges from "nothing" to "hard real-time" and the middle is where real systems actually live.

- **Asynchronous**: no bound on message delay, no bound on process speed, no reliable notion of time. Safety can be proven; liveness generally cannot (FLP).
- **Synchronous**: known upper bounds on message delay, clock drift, and relative process speed. Powerful enough to solve consensus trivially, but unrealistic — no cross-datacenter network gives hard bounds.
- **Partial synchrony**: bounds exist but are unknown, or are known and hold only *eventually* / *most of the time*. This is where Raft, Paxos, every gossip-based failure detector, and most practical distributed databases operate. Liveness is guaranteed only when synchrony holds; safety is preserved always.

The practical distinction matters because your protocol's guarantees are a function of the model. A Raft cluster across a WAN partition can lose liveness (no leader elected) but must not lose safety (no two conflicting commits). An "async" claim in a paper usually means the safety analysis is async-safe even though the liveness proof assumes partial synchrony.

```mermaid
graph LR
    Async[Asynchronous<br/>no timing bounds<br/>FLP applies] -->|add eventual bounds| Partial[Partial synchrony<br/>bounds hold eventually<br/>or most of the time]
    Partial -->|add hard bounds| Sync[Synchronous<br/>known upper bound on<br/>delay, drift, speed]

    Async --> AsyncCan[Can solve:<br/>reliable broadcast with failure detector<br/>self-stabilizing protocols]
    Partial --> PartialCan[Can solve:<br/>Raft/Paxos consensus<br/>failure detection<br/>leader election]
    Sync --> SyncCan[Can solve:<br/>trivial consensus<br/>synchronous BFT]
```

### Synchrony model comparison

| Assumption | Async | Partial sync | Sync |
|------------|:-----:|:------------:|:----:|
| Upper bound on message delay | no | eventually | yes |
| Upper bound on clock drift | no | eventually | yes |
| Upper bound on relative process speed | no | eventually | yes |
| Timeouts meaningful? | no | yes (but may fire wrongly) | yes |
| Consensus solvable? | **no (FLP)** | yes (when sync holds) | yes |
| Realistic for production? | too weak | **yes** | too strong |
| Examples of protocols | pure gossip for rumor spreading | Raft, Paxos, SWIM, HLC | lockstep simulations, FPGAs on same clock |

---

## 7. Failure Models

How a process can deviate from its spec defines how much machinery you need to tolerate it. The four classical models form a strict inclusion hierarchy — each one is a superset of the previous.

- **Crash-stop**: process halts and never returns. Simplest to reason about; algorithms here are correctness-easy, just detection-hard.
- **Crash-recovery**: process halts and later comes back, possibly with some durable state. Adds protocols for re-integration (log replay, state transfer) and introduces a distinct identity problem (is the revived process "the same one" for the purposes of quorum?). From the *outside*, looks like an omission during the downtime.
- **Omission**: process skips steps, drops messages, or executes them invisibly. Captures network partitions, slow nodes, dropped RPCs, overflowing queues. Crash-stop is a degenerate omission (omits everything forever).
- **Arbitrary / Byzantine**: process can do anything, including actively lying, forging messages, or colluding with other faulty processes. Requires cryptographic signatures, majority-of-three quorum intersection, and usually `3f + 1` nodes to tolerate `f` failures.

Each stronger model requires strictly more redundancy. Crash-stop tolerance needs `2f + 1` nodes for a majority quorum. Byzantine tolerance needs `3f + 1` because any two quorums must intersect in at least `f + 1` correct nodes — otherwise liars can collude to present a split-brain view.

```mermaid
graph TB
    subgraph Hierarchy[Failure model hierarchy - each contains the previous]
        Crash[Crash-stop<br/>halts forever<br/>tolerance: 2f+1 nodes]
        Recov[Crash-recovery<br/>halts, returns with durable state<br/>+ recovery protocol]
        Omit[Omission<br/>drops messages<br/>looks like network partition]
        Byz[Byzantine / Arbitrary<br/>adversarial, lies, forges<br/>tolerance: 3f+1 nodes + crypto]
    end
    Crash --> Recov
    Recov --> Omit
    Omit --> Byz

    Crash -.->|example| Ex1[Classic DBs<br/>Raft, Paxos quorums]
    Recov -.->|example| Ex2[ARIES-style recovery<br/>replica re-integration]
    Omit -.->|example| Ex3[CAP partitions<br/>TCP session drops]
    Byz -.->|example| Ex4[Blockchains PBFT<br/>aerospace quad-redundant]
```

### Failure-model tolerance costs

| Model | What fails | Detection difficulty | Min nodes for f faults | Extra machinery needed |
|-------|-----------|----------------------|:----------------------:|------------------------|
| Crash-stop | Process halts, no more messages | Hard (looks like slow peer in async) | `2f + 1` | Timeouts, failure detector |
| Crash-recovery | Halts then returns | Identity question on return | `2f + 1` | Durable log, recovery protocol, CLRs |
| Omission | Skips or drops messages | Indistinguishable from partition | `2f + 1` | Retries, idempotence, reconciliation |
| Byzantine | Arbitrary / adversarial behaviour | Very hard — can actively mimic correctness | `3f + 1` | Digital signatures, MACs, cross-validation, quorum intersection proofs |

---

## Trade-offs: What Failure-Model Assumption Costs You

The model you assume is a direct knob on latency, hardware cost, and protocol complexity. Assume too little and you pay for protection you don't need; assume too much and your system silently misbehaves when reality violates the assumption.

| Assumption | Extra cost over crash-stop | When to pay it | Canonical systems |
|-----------|---------------------------|-----------------|-------------------|
| Crash-stop only | baseline | Trusted hardware, trusted operators, internal DC | Most classical databases, Raft, Paxos |
| Crash-recovery | +durable state + replay protocol | Long-running processes, expected maintenance windows | ARIES, most replicated DBs |
| Omission-tolerant | +retry / timeout / reconciliation logic | WAN replication, unreliable networks | Dynamo, Cassandra, any partition-tolerant system |
| Byzantine-tolerant | +signatures (~100 us per verify), 3f+1 nodes instead of 2f+1, larger quorums | Untrusted operators, mutually-distrusting nodes, high-stakes correctness | Blockchains, PBFT-based DBs (BFT-SMaRt), aerospace quad-redundant |

---

## Key Takeaways

1. **Distributed systems are defined by their failure models, not their features.** The right question is never "what does it do?" but "what assumptions does it make about the network, clocks, and processes — and what happens when they fail?"
2. **The Two Generals' Problem is the ceiling.** You cannot get common knowledge over a lossy asynchronous channel via any finite protocol. Every real "reliable" system is quietly settling for approximate common knowledge.
3. **FLP Impossibility fences the async/crash intersection.** Pure async + possible crash + deterministic + always-terminating consensus is a contradiction. Real systems relax async (partial sync), determinism (randomization), or certainty (failure detectors).
4. **Partial synchrony is the realistic model.** Almost every production protocol (Raft, Paxos, SWIM, gossip failure detectors) assumes bounds hold "eventually" or "most of the time," and delivers safety always but liveness only when the model holds.
5. **The link abstraction ladder is universal.** Fair-loss → stubborn → perfect, built from ACKs, seqnos, retransmits, and dedup. Every reliable messaging system in industry is a specific point on this ladder.
6. **Exactly-once is an illusion you create at the receiver.** Links cannot deliver exactly once; the trick is at-least-once transmission + receiver-side dedup. Don't argue about terminology — ask where the dedup lives.
7. **Failure-model cost grows non-linearly.** Crash-stop tolerance is ~free; Byzantine tolerance demands `3f + 1` nodes, cryptographic signatures on every message, and dramatically more complex quorum logic. Don't buy a stronger model than your adversary justifies.

## See Also

- [[01-fallacies-of-distributed-computing]]
- [[02-partial-failures-and-cascading-failures]]
- [[03-link-abstractions-and-delivery-semantics]]
- [[04-two-generals-problem]]
- [[05-flp-impossibility-and-consensus]]
- [[06-system-synchrony-models]]
- [[07-failure-models]]